# EmporiUm Book Store
I am anazlyzing sales manager Miami Vue and Bo Heap, in the NorthEast region. For states New Jersey(Miami Vue) and Massachusetts(Bo Heap). New Jersey has 16 locations and Massachusetts has 18 store locations. 

In [21]:
import pandas as pd
import numpy as np

In [22]:
df1= pd.read_csv('customer_list.csv')

In [23]:
df1.info()
#521 non null 
#dtype: 1 string
#1 column

<class 'pandas.DataFrame'>
RangeIndex: 521 entries, 0 to 520
Data columns (total 1 columns):
 #   Column                                           Non-Null Count  Dtype
---  ------                                           --------------  -----
 0   cust_id|date|time|name|email|phone|sms-opt-out   521 non-null    str  
dtypes: str(1)
memory usage: 4.2 KB


In [24]:
df2 = pd.read_csv('ProductCategories.csv')

In [25]:
df2.info()
#52 non null
#dtype:1 int54, 3 string
#4 columns

<class 'pandas.DataFrame'>
RangeIndex: 52 entries, 0 to 51
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   CategoryID     52 non-null     int64
 1   Category       52 non-null     str  
 2   SubcategoryID  52 non-null     str  
 3   Subcategory    52 non-null     str  
dtypes: int64(1), str(3)
memory usage: 1.8 KB


In [26]:
df3 = pd.read_csv('Products.csv')

In [27]:
df3.info()
#669 non null
#4 columns
#dtype:1 int64, 3 string

<class 'pandas.DataFrame'>
RangeIndex: 669 entries, 0 to 668
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   Prod Num       669 non-null    str  
 1   Product        669 non-null    str  
 2   CategoryID     669 non-null    int64
 3   SubcategoryID  669 non-null    str  
dtypes: int64(1), str(3)
memory usage: 21.0 KB


In [28]:
df4 = pd.read_csv('StoreDetail.csv')

In [29]:
df4.info()
#111 non null
#6 columns
#dtype:1 int64, 5 string

<class 'pandas.DataFrame'>
RangeIndex: 111 entries, 0 to 110
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Store Location     111 non-null    str  
 1   State              111 non-null    str  
 2   Store ID           111 non-null    int64
 3   Territory Manager  111 non-null    str  
 4   Region             111 non-null    str  
 5   Region Director    111 non-null    str  
dtypes: int64(1), str(5)
memory usage: 5.3 KB


In [30]:
df5 = pd.read_csv('StoreSales.csv')

In [31]:
df5.info()
#335129 non null
#5 columns
#dtype:1 int64, 2 floats64, 2 strings 

<class 'pandas.DataFrame'>
RangeIndex: 335129 entries, 0 to 335128
Data columns (total 5 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Transaction Date  335129 non-null  str    
 1   Store ID          335129 non-null  int64  
 2   RewardsID         34943 non-null   float64
 3   Prod Num          335129 non-null  str    
 4   Sale Amount       335129 non-null  float64
dtypes: float64(2), int64(1), str(2)
memory usage: 12.8 MB


# Core Marketing Analysis
The marketing manager wants to know:

Who are the territory managers for the sales territories assigned? What are the store IDs and cities
for the stores in each assigned sales territory?

In [ ]:
store_details = df4[df4['Territory Manager'].isin(['Miami Vue', 'Bo Heap'])][['Region', 'Territory Manager', 'Store ID', 'State', 'Store Location']]
store_details
#isin(), filters the row in territory manager to find Miami or Bo

,Region,Territory Manager,Store ID,State,Store Location
52,Northeast,Bo Heap,730,Massachusetts,Boston
53,Northeast,Bo Heap,801,Massachusetts,Attleboro
54,Northeast,Bo Heap,802,Massachusetts,Falmouth
55,Northeast,Bo Heap,803,Massachusetts,Framingham
56,Northeast,Bo Heap,804,Massachusetts,Haverhill
57,Northeast,Bo Heap,805,Massachusetts,Hingham
58,Northeast,Bo Heap,806,Massachusetts,Holyoke
59,Northeast,Bo Heap,807,Massachusetts,Leominster
60,Northeast,Bo Heap,808,Massachusetts,Lowell
61,Northeast,Bo Heap,809,Massachusetts,Lynn


What is monthly total revenue for in-store sales in each of the two sales territories, over the full
period covered by the data?

In [ ]:
from datetime import datetime
#to fix date 

In [70]:
#need to join merge df4 and df5 for store id 
merged_sales = pd.merge(df4, df5, on= 'Store ID')
# now find miami and bo using .isin() again to filter and find rows
terr_manager = ['Miami Vue', 'Bo Heap']
filtered_manager = merged_sales[merged_sales['Territory Manager'].isin(terr_manager)]
# to get the monthly i need to convert transaction date to year and month
# have to import datetime module use .to_datetime
date_fix = pd.to_datetime(filtered_manager['Transaction Date'])
#%Y full year 4 digit , %m as number 01-12
filtered_manager['Year-Month'] = date_fix.dt.strftime('%Y-%m')
# this only gave me total rev not monthly
#total_rev = filtered_manager.groupby('Territory Manager')['Sale Amount'].sum()
#use .sum() to get the total revenue 
monthly_rev = filtered_manager.groupby(['Territory Manager','Year-Month'])['Sale Amount'].sum()

monthly_rev


Territory Manager  Year-Month
Bo Heap            2022-01        69396.42
                   2022-02        65616.00
                   2022-03        77390.00
                   2022-04        81333.60
                   2022-05        75919.64
                                   ...    
Miami Vue          2025-08       151923.99
                   2025-09       163449.78
                   2025-10       250830.46
                   2025-11       146953.33
                   2025-12       170994.00
Name: Sale Amount, Length: 96, dtype: float64

How would you rank the sales performance of each store in each sales territory? Which are the
top-performing stores?

In [ ]:
#use rank ()
#join merge df4 and df5 
merged_sales = pd.merge(df4, df5, on= 'Store ID')
terr_manager = ['Miami Vue', 'Bo Heap']
filtered_manager = merged_sales[merged_sales['Territory Manager'].isin(terr_manager)]
#sum of each store
store_performance = filtered_manager.groupby(['Territory Manager', 'Store ID', 'Store Location'], as_index=False)['Sale Amount'].sum()
#ascending false = high to low
store_performance['Rank'] = store_performance.groupby('Territory Manager')['Sale Amount'].rank(ascending=False)
#sort by manager to rank using sort values for column names
store_performance = store_performance.sort_values(['Territory Manager', 'Rank'])
store_performance




,Territory Manager,Store ID,Store Location,Sale Amount,Rank
17,Bo Heap,817,Worcester,602183.44,1.0
7,Bo Heap,807,Leominster,338009.10,2.0
10,Bo Heap,810,Nantucket,335547.81,3.0
14,Bo Heap,814,Provincetown,328860.51,4.0
12,Bo Heap,812,Northampton,322039.24,5.0
6,Bo Heap,806,Holyoke,320516.53,6.0
16,Bo Heap,816,Somerville,312873.59,7.0
4,Bo Heap,804,Haverhill,305762.60,8.0
9,Bo Heap,809,Lynn,302049.65,9.0
13,Bo Heap,813,Pittsfield,301281.50,10.0


Comparing the customer ID from the customer list data with the rewards ID from the sales data,
who were the top customers in each sales territory?


In [ ]:
#join merge df5 and df1 reward id and cust id
# can only merge two tables at a time have to use left on and right on 
#since cust id and rewardsid dont have the same name 
# hard time with this one it a long string name for the column 
#have to use split to separte the string to columns .iloc for indexing
df1_split = df1.iloc[:, 0].str.split('|', expand=True)
# naming the columns individually
df1_split.columns = ['cust_id', 'date', 'time', 'name', 'email', 'phone', 'sms-opt-out']
#make sure both are integers for id column
#fillna to replace NaN with 0 vs missing value
df5['RewardsID'] = df5['RewardsID']. fillna(0).astype(int)
df1_split['cust_id'] = df1_split['cust_id'].astype(int)

merged_2= pd.merge(df1_split, df5, right_on='RewardsID', left_on='cust_id')
all_merged = pd.merge(merged_2, df4, on='Store ID')

# keep this from previous code to fliter manager
filtered_manager = all_merged[all_merged['Territory Manager'].isin(['Miami Vue', 'Bo Heap'])]

# total customer spending to find top customer
customer_money = filtered_manager.groupby(['Territory Manager', 'Store ID', 'cust_id', 'name'], as_index=False)['Sale Amount'].sum()

#rank the customers to find the top ascending false high to low
#sort to show top customer ranking
top_customer = customer_money.sort_values(by='Sale Amount', ascending=False)
#.head to show only the 10 customers i dont need the rest
top_customer.head(10)

,Territory Manager,Store ID,cust_id,name,Sale Amount
2536,Bo Heap,813,384,Tracy Jordan,3041.06
3015,Bo Heap,816,188,Mellie Grant,2532.56
2359,Bo Heap,812,412,Big Bird,2295.47
3558,Miami Vue,824,307,Angel,2218.26
5615,Miami Vue,835,161,Lieutenant Duffy,2181.66
3206,Bo Heap,817,85,Dar Adal,2090.51
6409,Miami Vue,839,123,Lady Elaine,2084.57
3474,Miami Vue,824,32,Marge,2022.28
5230,Miami Vue,833,306,Cordelia Chase,2021.22
1123,Bo Heap,805,492,Lucy Ricardo,2014.30


What is the number of transactions per month by product category in each assigned territory?
What is total sales revenue per month by category? What might this tell you about the most
popular products, and where could there be opportunity for growth?

In [139]:
#join merge df5 and df3 prodnum and df4
merged_categories = pd.merge(df3, df5, on= 'Prod Num')
all_categories = pd.merge(merged_categories, df4, on='Store ID')
# keep this from previous code to fliter manager
filtered_manager = all_categories[all_categories['Territory Manager'].isin(['Miami Vue', 'Bo Heap'])]
#fix date to months
filtered_manager['Transaction Date'] = pd.to_datetime(filtered_manager['Transaction Date'])
#%Y full year 4 digit , %m month as number 01-12
filtered_manager['Year-Month'] =filtered_manager['Transaction Date'].dt.strftime('%Y-%m')


#transaction by categories monthly
transactions = filtered_manager.groupby(['Territory Manager', 'Year-Month', 'CategoryID'], as_index=False)['Transaction Date'].count()
#rename column to count 
transactions = transactions.rename(columns={'Transaction Date': 'Transaction Count'})
print(transactions)

#yearly = filtered_manager.groupby(['Territory Manager', 'Year-Month']).sum('Sale Amount')
#print(yearly)

#revenue per month by category
revenue = filtered_manager.groupby(['Territory Manager', 'Year-Month', 'CategoryID'], as_index=False)['Sale Amount'].sum()
print(revenue)


#top revnue category id by manager
miami_top = revenue[revenue['Territory Manager'] == 'Miami Vue'].sort_values(by='Sale Amount', ascending=False)
print(miami_top.head(5))
#.tail for bottom 5

bo_top = revenue[revenue['Territory Manager'] == 'Bo Heap'].sort_values(by='Sale Amount', ascending=False)
print(bo_top.head(5))





    Territory Manager Year-Month  CategoryID  Transaction Count
0             Bo Heap    2022-01         100                 73
1             Bo Heap    2022-01         110                106
2             Bo Heap    2022-01         115                 93
3             Bo Heap    2022-01         120                112
4             Bo Heap    2022-01         125                 41
..                ...        ...         ...                ...
571         Miami Vue    2025-12         110                227
572         Miami Vue    2025-12         115                189
573         Miami Vue    2025-12         120                281
574         Miami Vue    2025-12         125                 78
575         Miami Vue    2025-12         130                257

[576 rows x 4 columns]
    Territory Manager Year-Month  CategoryID  Sale Amount
0             Bo Heap    2022-01         100     12333.33
1             Bo Heap    2022-01         110      1138.12
2             Bo Heap    2022-01  

What is your recommendation for where to focus marketing attention in the next quarter?

For the final question seeking a recommendation of where to focus marketing attention in the next
quarter?